# English pronouns — data creation, training, and options

This notebook walks through the **pronoun workflow** step by step: first we create training and neutral data from Wikipedia using `TextPredictionDataCreator`, then we train a GRADIEND model on singular vs plural (3SG: he/she/it vs 3PL: they). Finally, we show how to use `class_merge_map` to learn number across all pronouns (singular vs plural).

**What you'll do:**
1. Create pronoun data from Wikipedia (1SG, 1PL, 2, 3SG, 3PL)
2. Train on 3SG vs 3PL
3. Evaluate encoder and decoder
4. (Optional) Use `class_merge_map` to learn singular vs plural from all pronouns

**Requires:** `pip install gradiend[data] transformers torch datasets`

## 1. Create data with TextPredictionDataCreator

We use Wikipedia as base data and filter sentences that contain pronouns. Each pronoun group (1SG, 1PL, 2, 3SG, 3PL) gets its own class. The creator masks pronouns and produces training data in per-class format. Neutral data excludes all pronouns—useful for language-model evaluation.

In [ ]:
from pathlib import Path
from gradiend import TextFilterConfig, TextPreprocessConfig, TextPredictionDataCreator

DATA_DIR = "data/english_pronouns"
NEUTRAL_EXCLUDE = ["i", "we", "you", "he", "she", "it", "they", "me", "us", "him", "her", "them"]

creator = TextPredictionDataCreator(
    base_data="wikipedia",
    text_column="text",
    hf_config="20220301.en",
    preprocess=TextPreprocessConfig(
        split_to_sentences=True,
        min_chars=10,
        max_chars=200,
    ),
    trust_remote_code=True,
    spacy_model="en_core_web_sm",
    feature_targets=[
        TextFilterConfig(target="I", spacy_tags={"pos": "PRON"}, id="1SG"),
        TextFilterConfig(target="we", spacy_tags={"pos": "PRON"}, id="1PL"),
        TextFilterConfig(target="you", spacy_tags={"pos": "PRON"}, id="2"),
        TextFilterConfig(targets=["he", "she", "it"], spacy_tags={"pos": "PRON"}, id="3SG"),
        TextFilterConfig(target="they", spacy_tags={"pos": "PRON"}, id="3PL"),
    ],
    output_dir=DATA_DIR,
)

training = creator.generate_training_data(max_size_per_class=100, format="per_class", balance="try")
neutral = creator.generate_neutral_data(additional_excluded_words=NEUTRAL_EXCLUDE, max_size=100)
print("Training classes:", list(training.keys()))
print("Neutral rows:", len(neutral))

## 2. Train a GRADIEND model (3SG vs 3PL)

We train on **3SG** (he/she/it) vs **3PL** (they)—the simplest singular–plural contrast. Pass the created `training` and `neutral` data directly to the trainer. With `target_classes=["3SG", "3PL"]`, GRADIEND learns to separate these two classes from their gradients.

In [ ]:
from gradiend import TextPredictionTrainer, TextPredictionConfig, TrainingArguments

config = TextPredictionConfig(
    run_id="pronoun_3sg_3pl",
    data=training,
    target_classes=["3SG", "3PL"],
    eval_neutral_data=neutral,
)

args = TrainingArguments(
    experiment_dir="runs/notebook_english_pronouns",
    train_batch_size=8,
    eval_steps=100,
    num_train_epochs=2,
    max_steps=500,
    source="factual",
    target="diff",
    eval_batch_size=4,
    learning_rate=1e-4,
)

trainer = TextPredictionTrainer(model="bert-base-uncased", config=config, args=args)
trainer.train()
trainer.plot_training_convergence()

## 3. Evaluate encoder and decoder

Encoder evaluation shows how well the model separates 3SG and 3PL. Decoder evaluation finds the best learning rate and feature factor to shift the base model toward one class.

In [ ]:
trainer.evaluate_encoder(plot=True)
dec = trainer.evaluate_decoder()
for k, v in dec.get("summary", {}).items():
    if isinstance(v, dict) and "value" in v:
        print(f"  {k}: {v.get('value')}")

# Optional: get a model biased toward 3SG
# changed_model = trainer.rewrite_base_model(decoder_results=dec, target_class="3SG")

## 4. (Optional) Learn singular vs plural across all pronouns

Instead of 3SG vs 3PL only, you can merge classes to learn **number** (singular vs plural) across all pronouns:

- **singular**: 1SG (I) + 3SG (he/she/it)
- **plural**: 1PL (we) + 3PL (they)

Use `class_merge_map` in the config. With exactly two merged classes, `target_classes` is inferred automatically.

In [ ]:
# Uncomment to train on merged singular vs plural:
# config_merged = TextPredictionConfig(
#     run_id="pronoun_singular_plural",
#     data=training,
#     eval_neutral_data=neutral,
#     class_merge_map={"singular": ["1SG", "3SG"], "plural": ["1PL", "3PL"]},
# )
# trainer_merged = TextPredictionTrainer(model="bert-base-uncased", config=config_merged, args=args)
# trainer_merged.train()

## Next steps

- **[Start here](https://aieng-lab.github.io/gradiend/start/)** — Minimal workflow with artificial texts.
- **[Data generation tutorial](https://aieng-lab.github.io/gradiend/tutorials/data-generation/)** — Syncretism, spaCy, morphology.